# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-demesa/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

I will rank pages for content refresh based on observable search performance signals. Pages with many impressions, few clicks, and a poor average search position receive a higher score because they may have the greatest opportunity to improve through content updates. The rule uses only information available at the decision time and does not rely on future outcomes.

## Reason codes

REFRESH_HIGH_IMPRESSIONS_LOW_CLICKS
- High search impressions but relatively few clicks.

REFRESH_LOW_CTR
- The page appears frequently in search results but attracts relatively few clicks.

REFRESH_LOW_VISIBILITY
- The page has a poor average search position and may benefit from content improvements.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
!git clone https://github.com/a-demesa/flyrank-ml.git

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 150 (delta 60), reused 86 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 1.85 MiB | 10.49 MiB/s, done.
Resolving deltas: 100% (60/60), done.


In [8]:
%cd flyrank-ml

/content/flyrank-ml/flyrank-ml


In [9]:
!pip -q install datasets huggingface_hub pandas

In [11]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

train = dataset["train"]

print(train)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


In [13]:
from pathlib import Path
import pandas as pd

# Take a manageable sample for the baseline rule
sample_df = train.select(range(10000)).to_pandas()

# Avoid division by zero
sample_df["ctr"] = sample_df["gsc_clicks"] / sample_df["gsc_impressions"].replace(0, 1)

# Baseline score
sample_df["baseline_score"] = (
    sample_df["gsc_impressions"] * 0.5
    + (1 - sample_df["ctr"]) * 100
    + sample_df["gsc_avg_position"] * 2
)

# Reason code
sample_df["reason_code"] = "REFRESH_OPPORTUNITY"

# Action label
sample_df["action"] = "Refresh Content"

# Rank pages
ranked = sample_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# Create output folder
Path("work/outputs").mkdir(parents=True, exist_ok=True)

# Save CSV
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully!")

ranked.head(20)

CSV written successfully!


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,ctr,baseline_score,reason_code,action
0,2025-02-12,client_73cda7b4e4f265ea,content_4e8d1e11f60fe6ba,True,True,True,False,506,11,3565.0,...,0,0,0,0,0,0,0.021739,364.916996,REFRESH_OPPORTUNITY,Refresh Content
1,2025-02-11,client_73cda7b4e4f265ea,content_6ac06aec2173eacb,True,True,True,False,1,0,127.0,...,0,0,0,0,0,0,0.000000,354.500000,REFRESH_OPPORTUNITY,Refresh Content
2,2025-02-13,client_73cda7b4e4f265ea,content_4e8d1e11f60fe6ba,True,True,True,False,466,2,3495.0,...,0,0,0,0,0,0,0.004292,347.570815,REFRESH_OPPORTUNITY,Refresh Content
3,2025-02-13,client_73cda7b4e4f265ea,content_df4fd5ef66c78886,True,True,True,False,202,0,14275.0,...,0,0,0,0,0,0,0.000000,342.336634,REFRESH_OPPORTUNITY,Refresh Content
4,2025-02-13,client_73cda7b4e4f265ea,content_179e7bf24d8d7530,True,True,True,False,167,0,13140.0,...,0,0,0,0,0,0,0.000000,340.865269,REFRESH_OPPORTUNITY,Refresh Content
5,2025-02-10,client_9958f0a7ae1df715,content_f5950be18c9f27db,True,True,True,False,165,0,12706.0,...,0,0,0,0,0,0,0.000000,336.512121,REFRESH_OPPORTUNITY,Refresh Content
6,2025-02-11,client_73cda7b4e4f265ea,content_516bb6f195d4ef04,True,True,True,False,1,0,117.0,...,0,0,0,0,0,0,0.000000,334.500000,REFRESH_OPPORTUNITY,Refresh Content
7,2025-02-10,client_ff644d8251367cbb,content_8cd7bbc2fdd9b947,True,True,True,False,153,0,11900.0,...,0,0,0,0,0,0,0.000000,332.055556,REFRESH_OPPORTUNITY,Refresh Content
8,2025-02-12,client_73cda7b4e4f265ea,content_df4fd5ef66c78886,True,True,True,False,159,0,11087.0,...,0,0,0,0,0,0,0.000000,318.959119,REFRESH_OPPORTUNITY,Refresh Content
9,2025-02-13,client_73cda7b4e4f265ea,content_7f472fe194aa5ba5,True,True,True,False,113,0,8948.0,...,0,0,0,0,0,0,0.000000,314.871681,REFRESH_OPPORTUNITY,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

| Rank | Action | Reason Code | Confidence Note | What would make it wrong |
|------|--------|-------------|-----------------|--------------------------|
| 1 | Refresh Content | REFRESH_OPPORTUNITY | High impressions with low CTR suggest opportunity. | Seasonal traffic changes may explain the pattern. |
| 2 | Refresh Content | REFRESH_OPPORTUNITY | High impressions with below-average CTR. | Recent content updates may not yet be reflected. |
| 3 | Refresh Content | REFRESH_OPPORTUNITY | Poor average position despite visibility. | The page may target highly competitive keywords. |
| 4 | Refresh Content | REFRESH_OPPORTUNITY | Strong search exposure but limited clicks. | Low search intent could explain the CTR. |
| 5 | Refresh Content | REFRESH_OPPORTUNITY | Multiple signals indicate refresh potential. | External events may temporarily affect performance. |
| 6 | Refresh Content | REFRESH_OPPORTUNITY | High opportunity based on search metrics. | Missing analytics data could affect interpretation. |
| 7 | Refresh Content | REFRESH_OPPORTUNITY | Ranking suggests optimization opportunity. | The page may already be scheduled for updates. |
| 8 | Refresh Content | REFRESH_OPPORTUNITY | Search visibility exceeds engagement. | Keyword intent may not match the content. |
| 9 | Refresh Content | REFRESH_OPPORTUNITY | Consistent low CTR with measurable impressions. | Recent algorithm changes could influence rankings. |
| 10 | Refresh Content | REFRESH_OPPORTUNITY | Baseline score indicates refresh priority. | Additional business context may change the decision. |
| 11 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Limited historical context. |
| 12 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Temporary traffic fluctuations. |
| 13 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Missing context outside search metrics. |
| 14 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Business priorities may differ. |
| 15 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Seasonal effects. |
| 16 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Competition changes. |
| 17 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Data availability limitations. |
| 18 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Content recently updated. |
| 19 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Search behavior changes. |
| 20 | Refresh Content | REFRESH_OPPORTUNITY | High baseline score. | Additional manual review recommended. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

Some pages may receive a high baseline score because of temporary traffic patterns or seasonal effects rather than genuine refresh opportunities. These cases should be manually reviewed before action is taken.

The baseline rule uses only observable search performance signals available at the decision time. No future performance data, labels, or outcome-derived fields were used, helping avoid data leakage.

### Weak picks

Some lower-confidence recommendations may have:
- High impressions because of seasonal or temporary trends.
- Low CTR caused by search intent rather than poor content.
- Poor average position due to highly competitive keywords instead of content quality.

These pages should be reviewed manually before deciding to refresh them.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.